<a href="https://colab.research.google.com/github/Ayan2582/Delineation-of-RTS/blob/main/Mask_r_cnn_base_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("Pre-install torch version:", torch.__version__, "| CUDA:", torch.version.cuda)
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
print("Detectron2 installed. Now RESTART THE RUNTIME (Runtime -> Restart runtime), then continue from the next cell.")

In [ ]:
import os
import copy
import json
import random
import numpy as np
import detectron2
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_train_loader, build_detection_test_loader
from detectron2.data import detection_utils as utils
from detectron2.data import transforms as T
from detectron2.data.dataset_mapper import DatasetMapper
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.structures import BoxMode
from pycocotools.coco import COCO

print("Detectron2 version:", detectron2.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "instances_train.json" in files:
        print("FOUND:")
        print(os.path.join(root, "instances_train.json"))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT = "/content/drive/MyDrive/competition_release"
OUTPUT_DIR = "/content/drive/MyDrive/geoai_arctic_checkpoints_d2_r101"

TRAIN_IMAGES_DIR = os.path.join(DATASET_ROOT, "train", "images")
TRAIN_ANN_PATH = os.path.join(DATASET_ROOT, "train", "annotations", "instances_train.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Images dir:", TRAIN_IMAGES_DIR)
print("Annotations:", TRAIN_ANN_PATH)
print("Output/checkpoint dir:", OUTPUT_DIR)

In [ ]:
def load_rgb(npz_path):
    data = np.load(npz_path)
    image = data["image"][:, :, :3].astype(np.float32)
    image = np.nan_to_num(image, nan=0.0, posinf=255.0, neginf=0.0)
    image = np.clip(image, 0.0, 255.0)
    return image

In [ ]:
def build_arctic_dataset_dicts(coco, image_ids, images_dir):
    dataset_dicts = []
    for image_id in image_ids:
        img_info = coco.loadImgs(image_id)[0]
        file_name = img_info["file_name"]
        if file_name.startswith("images/"):
            file_name = file_name[len("images/"):]
        record = {
            "file_name": os.path.join(images_dir, file_name),
            "image_id": image_id,
            "height": img_info["height"],
            "width": img_info["width"],
        }
        ann_ids = coco.getAnnIds(imgIds=image_id)
        anns = coco.loadAnns(ann_ids)
        objs = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue
            objs.append({
                "bbox": ann["bbox"],
                "bbox_mode": BoxMode.XYWH_ABS,
                "segmentation": ann["segmentation"],
                "category_id": 0,   # Detectron2 zero-based index — DO NOT change for training data
            })
        record["annotations"] = objs
        dataset_dicts.append(record)
    return dataset_dicts

Spitting the image for training

In [ ]:
full_coco = COCO(TRAIN_ANN_PATH)
all_image_ids = sorted(full_coco.getImgIds())
assert len(all_image_ids) == 756, (f"Expected 756 images, found {len(all_image_ids)}.")

rng = np.random.RandomState(42)
shuffled_ids = all_image_ids.copy()
rng.shuffle(shuffled_ids)

train_ids = shuffled_ids[:604]
val_ids = shuffled_ids[604:]

print("Train images:", len(train_ids))
print("Val images:", len(val_ids))

for name in ["arctic_rts_train", "arctic_rts_val"]:
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)

DatasetCatalog.register("arctic_rts_train", lambda: build_arctic_dataset_dicts(full_coco,train_ids, TRAIN_IMAGES_DIR))
DatasetCatalog.register("arctic_rts_val", lambda: build_arctic_dataset_dicts( full_coco, val_ids, TRAIN_IMAGES_DIR))
MetadataCatalog.get("arctic_rts_train").set( thing_classes=["RTS"])
MetadataCatalog.get("arctic_rts_val").set(thing_classes=["RTS"])
print("Registered datasets:", [d for d in DatasetCatalog.list() if "arctic_rts" in d])

preprocessing

In [ ]:
class ArcticNpzMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_rgb(dataset_dict["file_name"])
        utils.check_image_size(dataset_dict, image)
        aug_input = T.AugInput(image)
        transforms = self.augmentations(aug_input)
        image = aug_input.image
        image_shape = image.shape[:2]
        dataset_dict["image"] = torch.as_tensor(np.ascontiguousarray(image.transpose(2, 0, 1)))
        if not self.is_train:
            dataset_dict.pop("annotations", None)
            return dataset_dict
        annos = [
            utils.transform_instance_annotations(obj, transforms, image_shape)
            for obj in dataset_dict.pop("annotations")
            if obj.get("iscrowd", 0) == 0
        ]
        instances = utils.annotations_to_instances(annos, image_shape, mask_format="bitmask")
        dataset_dict["instances"] = utils.filter_empty_instances(instances)
        return dataset_dict

def build_train_augmentations():
    return [T.RandomFlip(horizontal=True, vertical=False, prob=0.5)]

def build_test_augmentations():
    return []

initilize

In [ ]:
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml")
cfg.DATASETS.TRAIN = ("arctic_rts_train",)
cfg.DATASETS.TEST = ("arctic_rts_val",)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.INPUT.FORMAT = "RGB"
cfg.MODEL.PIXEL_MEAN = [123.675, 116.280, 103.530]
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg.OUTPUT_DIR = OUTPUT_DIR

print("Backbone:", cfg.MODEL.RESNETS.DEPTH)
print("Number of classes:", cfg.MODEL.ROI_HEADS.NUM_CLASSES)
print("Device:", cfg.MODEL.DEVICE)

In [ ]:
ITERS_PER_EPOCH = len(train_ids) // 2
NUM_EPOCHS = 10
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 1e-4
cfg.SOLVER.WEIGHT_DECAY = 1e-4
cfg.SOLVER.MAX_ITER = ITERS_PER_EPOCH * NUM_EPOCHS
cfg.SOLVER.STEPS = (ITERS_PER_EPOCH * 3, ITERS_PER_EPOCH * 6, ITERS_PER_EPOCH * 9)
cfg.SOLVER.GAMMA = 0.1
cfg.SOLVER.WARMUP_ITERS = 100
cfg.SOLVER.CHECKPOINT_PERIOD = ITERS_PER_EPOCH
cfg.TEST.EVAL_PERIOD = ITERS_PER_EPOCH

print("Iterations per epoch:", ITERS_PER_EPOCH)
print("Max iterations", cfg.SOLVER.MAX_ITER)

In [ ]:
class ArcticTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        mapper = ArcticNpzMapper(cfg, is_train=True, augmentations=build_train_augmentations())
        return build_detection_train_loader(cfg, mapper=mapper)

    @classmethod
    def build_test_loader(cls, cfg, dataset_name):
        mapper = ArcticNpzMapper(cfg, is_train=False, augmentations=build_test_augmentations())
        return build_detection_test_loader(cfg, dataset_name, mapper=mapper)

    @classmethod
    def build_optimizer(cls, cfg, model):
        params = [p for p in model.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=cfg.SOLVER.BASE_LR, weight_decay=cfg.SOLVER.WEIGHT_DECAY)

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, tasks=("segm",), output_dir=output_folder)

In [ ]:
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print("Starting training...")
print("Output directory:", cfg.OUTPUT_DIR)
print("Total iterations:", cfg.SOLVER.MAX_ITER)

trainer = ArcticTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:

FINAL_MODEL_PATH = os.path.join(
    cfg.OUTPUT_DIR,
    "maskrcnn_rts_rgb_baseline_d2_resnet101_final.pth"
)

torch.save(
    trainer.model.state_dict(),
    FINAL_MODEL_PATH
)

print("Final model saved to:")
print(FINAL_MODEL_PATH)



In [ ]:
METRICS_PATH = os.path.join(OUTPUT_DIR, "metrics.json")

def load_best_checkpoint(output_dir, metrics_path):
    with open(metrics_path, "r") as f:
        lines = [json.loads(l) for l in f if l.strip()]

    ap_entries = [l for l in lines if "segm/AP" in l and "iteration" in l]
    if not ap_entries:
        raise RuntimeError("No 'segm/AP' entries found in metrics.json")

    best_entry = max(ap_entries, key=lambda l: l["segm/AP"])
    best_iter = best_entry["iteration"]
    best_ap = best_entry["segm/AP"]

    candidate = os.path.join(output_dir, f"model_{best_iter:07d}.pth")
    if not os.path.exists(candidate):
        available = sorted(
            int(f.split("_")[1].split(".")[0])
            for f in os.listdir(output_dir)
            if f.startswith("model_") and f.endswith(".pth") and f != "model_final.pth"
        )
        closest = min(available, key=lambda x: abs(x - best_iter))
        candidate = os.path.join(output_dir, f"model_{closest:07d}.pth")
        print(f"Exact checkpoint for iter {best_iter} not found — using closest available: {candidate}")

    return candidate, best_ap, best_iter

BEST_CHECKPOINT_PATH, BEST_VAL_AP, BEST_ITER = load_best_checkpoint(OUTPUT_DIR, METRICS_PATH)

print("Best checkpoint:")
print(os.path.basename(BEST_CHECKPOINT_PATH))
print("Best validation Mask AP:")
print(f"{BEST_VAL_AP:.2f}")

In [ ]:
import torch
import numpy as np
import os
import json
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import DatasetCatalog
from pycocotools import mask as mask_utils

# ============================================================
# 1. LOAD MODEL WITH COMPETITION AP50 STRICTNESS (50% THRESHOLD)
# ============================================================
cfg.MODEL.WEIGHTS = BEST_CHECKPOINT_PATH
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.50  # Aligned to AP50 criteria

eval_model = build_model(cfg)
eval_model.eval()
DetectionCheckpointer(eval_model).load(BEST_CHECKPOINT_PATH)

print("Loaded checkpoint for evaluation:")
print(BEST_CHECKPOINT_PATH)
print("Model is in eval() mode with a 50% confidence threshold.")


# ============================================================
# 2. ALIGNED IoU EVALUATION FUNCTION (WITH 0.0 PENALTY FOR MISSED SLUMPS)
# ============================================================
def calculate_iou(mask1, mask2):
    union = np.logical_or(mask1, mask2).sum()
    return np.logical_and(mask1, mask2).sum() / union if union > 0 else 0.0

def evaluate_mask_iou(dataset_name, prediction_dir):
    pred_file = os.path.join(prediction_dir, "coco_instances_results.json")
    with open(pred_file, "r") as f:
        predictions = json.load(f)

    pred_by_image = {}
    for p in predictions:
        pred_by_image.setdefault(p["image_id"], []).append(p)

    dataset = DatasetCatalog.get(dataset_name)
    all_ious = []
    total_gt = 0

    for record in dataset:
        h, w = record["height"], record["width"]

        # Decode Ground Truth Masks
        gt_masks = []
        for ann in record["annotations"]:
            seg = ann["segmentation"]
            rle = mask_utils.merge(mask_utils.frPyObjects(seg, h, w)) if isinstance(seg, list) else seg
            gt_masks.append(mask_utils.decode(rle).astype(bool))

        # Decode Predicted Masks
        pred_masks = []
        for pred in pred_by_image.get(record["image_id"], []):
            if "segmentation" in pred:
                pm = mask_utils.decode(pred["segmentation"])
                pred_masks.append((pm[:,:,0] if pm.ndim == 3 else pm).astype(bool))

        total_gt += len(gt_masks)

        # Greedy Matching (Missed ground truths get a 0.0 penalty)
        used_preds = set()
        for gt in gt_masks:
            best_iou, best_idx = 0.0, -1
            for idx, pred in enumerate(pred_masks):
                if idx in used_preds: continue
                iou = calculate_iou(gt, pred)
                if iou > best_iou:
                    best_iou, best_idx = iou, idx

            if best_idx != -1:
                used_preds.add(best_idx)
            all_ious.append(best_iou)

    all_ious = np.array(all_ious, dtype=np.float32)
    return {
        "Images": len(dataset),
        "GT Instances": total_gt,
        "Mean IoU": all_ious.mean(),
        "Median IoU": np.median(all_ious),
        "IoU >= 0.25 (%)": (all_ious >= 0.25).mean() * 100,
        "IoU >= 0.50 (%)": (all_ious >= 0.50).mean() * 100,
        "IoU >= 0.75 (%)": (all_ious >= 0.75).mean() * 100
    }


# ============================================================
# 3. RUN EVALUATION AND PRINT TABLE COMPARISON
# ============================================================
train_res = evaluate_mask_iou("arctic_rts_train", os.path.join(OUTPUT_DIR, "train_inference_bestckpt"))
val_res = evaluate_mask_iou("arctic_rts_val", os.path.join(OUTPUT_DIR, "val_inference_bestckpt"))

print(f"\n{'Metric':<22} | {'TRAIN':<12} | {'VALIDATION':<12}")
print("-" * 52)
for key in train_res.keys():
    t_val, v_val = train_res[key], val_res[key]
    if "IoU" in key:
        print(f"{key:<22} | {t_val:<12.2f} | {v_val:<12.2f}")
    else:
        print(f"{key:<22} | {int(t_val):<12} | {int(v_val):<12}")

In [ ]:
train_evaluator = COCOEvaluator(
    "arctic_rts_train",
    tasks=("segm",),
    output_dir=os.path.join(OUTPUT_DIR, "train_inference_bestckpt"),
)
train_loader = ArcticTrainer.build_test_loader(cfg, "arctic_rts_train")
train_results = inference_on_dataset(eval_model, train_loader, train_evaluator)
train_segm = train_results["segm"]

print("===== TRAINING-SET METRICS (best checkpoint, weights frozen) =====")
print(f"AP   (IoU 0.50:0.95): {train_segm['AP']:.2f}")
print(f"AP50                : {train_segm['AP50']:.2f}")
print(f"AP75                : {train_segm['AP75']:.2f}")
print(f"APs  (Small)        : {train_segm['APs']:.2f}")
print(f"APm  (Medium)       : {train_segm['APm']:.2f}")
print(f"APl  (Large)        : {train_segm['APl']:.2f}")

In [ ]:
import matplotlib.pyplot as plt
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.utils.visualizer import Visualizer, ColorMode

model = eval_model
model.eval()

train_dataset = DatasetCatalog.get("arctic_rts_train")
metadata = MetadataCatalog.get("arctic_rts_train")
mapper = ArcticNpzMapper(cfg, is_train=False, augmentations=build_test_augmentations())

samples = random.sample(train_dataset, 6)

with torch.no_grad():
    for sample in samples:
        display_image = load_rgb(sample["file_name"])
        display_image = np.clip(display_image, 0, 255).astype(np.uint8)

        mapped = mapper(copy.deepcopy(sample))          # same preprocessing as eval, no resize
        model_input = [{
            "image": mapped["image"],
            "height": sample["height"],
            "width": sample["width"],
        }]
        outputs = model(model_input)[0]
        instances = outputs["instances"].to("cpu")       # raw scores, no artificial 0.5 cutoff

        gt_viz = Visualizer(display_image, metadata=metadata, scale=1.0, instance_mode=ColorMode.IMAGE)
        gt_out = gt_viz.draw_dataset_dict(sample)

        pred_viz = Visualizer(display_image, metadata=metadata, scale=1.0, instance_mode=ColorMode.IMAGE)
        pred_out = pred_viz.draw_instance_predictions(instances)

        plt.figure(figsize=(16, 6))
        plt.subplot(1, 2, 1); plt.imshow(gt_out.get_image()); plt.title(f"Ground Truth (train) — {sample['image_id']}"); plt.axis("off")
        plt.subplot(1, 2, 2); plt.imshow(pred_out.get_image()); plt.title(f"Prediction — {len(instances)} instances"); plt.axis("off")
        plt.tight_layout(); plt.show()

        if len(instances) > 0:
            print("Confidence:", [round(x, 3) for x in instances.scores.numpy()])

In [ ]:
val_evaluator = COCOEvaluator(
    "arctic_rts_val",
    tasks=("segm",),
    output_dir=os.path.join(OUTPUT_DIR, "val_inference_bestckpt"),
)
val_loader = ArcticTrainer.build_test_loader(cfg, "arctic_rts_val")
val_results = inference_on_dataset(eval_model, val_loader, val_evaluator)
val_segm = val_results["segm"]

print("===== VALIDATION-SET METRICS (best checkpoint, weights frozen) =====")
print(f"AP   (IoU 0.50:0.95): {val_segm['AP']:.2f}")
print(f"AP50                : {val_segm['AP50']:.2f}")
print(f"AP75                : {val_segm['AP75']:.2f}")
print(f"APs  (Small)        : {val_segm['APs']:.2f}")
print(f"APm  (Medium)       : {val_segm['APm']:.2f}")
print(f"APl  (Large)        : {val_segm['APl']:.2f}")

In [ ]:
import matplotlib.pyplot as plt
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.utils.visualizer import Visualizer, ColorMode

model = eval_model
model.eval()

val_dataset = DatasetCatalog.get("arctic_rts_val")
metadata = MetadataCatalog.get("arctic_rts_val")
mapper = ArcticNpzMapper(cfg, is_train=False, augmentations=build_test_augmentations())

samples = random.sample(val_dataset, 6)

with torch.no_grad():
    for sample in samples:
        display_image = load_rgb(sample["file_name"])
        display_image = np.clip(display_image, 0, 255).astype(np.uint8)

        mapped = mapper(copy.deepcopy(sample))          # same preprocessing as eval, no resize
        model_input = [{
            "image": mapped["image"],
            "height": sample["height"],
            "width": sample["width"],
        }]
        outputs = model(model_input)[0]
        instances = outputs["instances"].to("cpu")       # raw scores, no artificial 0.5 cutoff

        gt_viz = Visualizer(display_image, metadata=metadata, scale=1.0, instance_mode=ColorMode.IMAGE)
        gt_out = gt_viz.draw_dataset_dict(sample)

        pred_viz = Visualizer(display_image, metadata=metadata, scale=1.0, instance_mode=ColorMode.IMAGE)
        pred_out = pred_viz.draw_instance_predictions(instances)

        plt.figure(figsize=(16, 6))
        plt.subplot(1, 2, 1); plt.imshow(gt_out.get_image()); plt.title(f"Ground Truth (val) — {sample['image_id']}"); plt.axis("off")
        plt.subplot(1, 2, 2); plt.imshow(pred_out.get_image()); plt.title(f"Prediction — {len(instances)} instances"); plt.axis("off")
        plt.tight_layout(); plt.show()

        if len(instances) > 0:
            print("Confidence:", [round(x, 3) for x in instances.scores.numpy()])

In [ ]:
print("===== TRAIN vs VAL =====")
print(f"Train AP: {train_segm['AP']:.2f}   |   Val AP: {val_segm['AP']:.2f}")

In [ ]:
import os
import json
import numpy as np
from detectron2.data import DatasetCatalog
from pycocotools import mask as mask_utils

def calculate_iou(mask1, mask2):
    union = np.logical_or(mask1, mask2).sum()
    return np.logical_and(mask1, mask2).sum() / union if union > 0 else 0.0

def evaluate_mask_iou(dataset_name, prediction_dir):
    # 1. Load Predictions
    pred_file = os.path.join(prediction_dir, "coco_instances_results.json")
    with open(pred_file, "r") as f:
        predictions = json.load(f)

    pred_by_image = {}
    for p in predictions:
        pred_by_image.setdefault(p["image_id"], []).append(p)

    # 2. Evaluate Dataset
    dataset = DatasetCatalog.get(dataset_name)
    all_ious = []
    total_gt = 0

    for record in dataset:
        h, w = record["height"], record["width"]

        # Decode Ground Truth
        gt_masks = []
        for ann in record["annotations"]:
            seg = ann["segmentation"]
            rle = mask_utils.merge(mask_utils.frPyObjects(seg, h, w)) if isinstance(seg, list) else seg
            gt_masks.append(mask_utils.decode(rle).astype(bool))

        # Decode Predictions
        pred_masks = []
        for pred in pred_by_image.get(record["image_id"], []):
            if "segmentation" in pred:
                pm = mask_utils.decode(pred["segmentation"])
                pred_masks.append((pm[:,:,0] if pm.ndim == 3 else pm).astype(bool))

        total_gt += len(gt_masks)

        # 3. Greedy Matching (Missed GTs append a 0.0 penalty)
        used_preds = set()
        for gt in gt_masks:
            best_iou, best_idx = 0.0, -1
            for idx, pred in enumerate(pred_masks):
                if idx in used_preds: continue
                iou = calculate_iou(gt, pred)
                if iou > best_iou:
                    best_iou, best_idx = iou, idx

            if best_idx != -1:
                used_preds.add(best_idx)
            all_ious.append(best_iou)

    # 4. Calculate Summary
    all_ious = np.array(all_ious, dtype=np.float32)
    return {
        "Images": len(dataset),
        "GT Instances": total_gt,
        "Mean IoU": all_ious.mean(),
        "Median IoU": np.median(all_ious),
        "IoU >= 0.25 (%)": (all_ious >= 0.25).mean() * 100,
        "IoU >= 0.50 (%)": (all_ious >= 0.50).mean() * 100,
        "IoU >= 0.75 (%)": (all_ious >= 0.75).mean() * 100
    }

# ============================================================
# EXECUTE & PRINT COMPARISON
# ============================================================
train_res = evaluate_mask_iou("arctic_rts_train", os.path.join(OUTPUT_DIR, "train_inference_bestckpt"))
val_res = evaluate_mask_iou("arctic_rts_val", os.path.join(OUTPUT_DIR, "val_inference_bestckpt"))

print(f"\n{'Metric':<22} | {'TRAIN':<12} | {'VALIDATION':<12}")
print("-" * 52)
for key in train_res.keys():
    t_val, v_val = train_res[key], val_res[key]

    if "IoU" in key:
        print(f"{key:<22} | {t_val:<12.2f} | {v_val:<12.2f}")
    else:
        print(f"{key:<22} | {int(t_val):<12} | {int(v_val):<12}")

In [ ]:
import pandas as pd
TEST_MANIFEST_PATH = os.path.join(DATASET_ROOT, "metadata", "test_manifest.csv")
TEST_IMAGES_DIR_CANDIDATES = [
    os.path.join(DATASET_ROOT, "test", "images"),
    os.path.join(DATASET_ROOT, "test_images"),
]

test_manifest = pd.read_csv(TEST_MANIFEST_PATH)
print("Test manifest columns:", list(test_manifest.columns))
print("Number of test chips:", len(test_manifest))

# --- Adjust these two lines if your manifest uses different column names ---
IMAGE_ID_COL = "image_id" if "image_id" in test_manifest.columns else test_manifest.columns[0]
FILE_COL = next((c for c in test_manifest.columns if "file" in c.lower() or "path" in c.lower()), None)
assert FILE_COL is not None, "Could not find a filename/path column in test_manifest.csv — set FILE_COL manually."

def resolve_test_path(file_name):
    if os.path.isabs(file_name) and os.path.exists(file_name):
        return file_name
    for base in TEST_IMAGES_DIR_CANDIDATES:
        candidate = os.path.join(base, os.path.basename(file_name))
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(f"Could not locate test chip: {file_name}")

test_records = []
for _, row in test_manifest.iterrows():
    path = resolve_test_path(str(row[FILE_COL]))
    test_records.append({"image_id": int(row[IMAGE_ID_COL]), "file_name": path})

print("Resolved", len(test_records), "hidden test image chips.")
print("test_records is kept separate from arctic_rts_train / arctic_rts_val.")
print("It is NOT added to cfg.DATASETS.TRAIN or cfg.DATASETS.TEST, and is never used for AP metrics.")

In [ ]:
from pycocotools import mask as mask_utils
SCORE_THRESH = 0.5
MAX_DETS_PER_IMAGE = 10

def run_inference_on_test(records, model, score_thresh, max_dets):
    predictions = []
    model.eval()
    with torch.no_grad():
        for rec in records:
            image = load_rgb(rec["file_name"])   # RGB baseline — first 3 bands only
            h, w = image.shape[:2]
            image_t = torch.as_tensor(np.ascontiguousarray(image.transpose(2, 0, 1)))
            outputs = model([{"image": image_t, "height": h, "width": w}])[0]
            instances = outputs["instances"].to("cpu")

            scores = instances.scores.numpy()
            keep = scores >= score_thresh
            instances = instances[keep]
            scores = instances.scores.numpy()

            if len(instances) > max_dets:
                top_idx = np.argsort(-scores)[:max_dets]
                instances = instances[top_idx]
                scores = instances.scores.numpy()

            masks = instances.pred_masks.numpy() if len(instances) > 0 else np.zeros((0, h, w), dtype=bool)

            for i in range(len(instances)):
                mask = np.asfortranarray(masks[i].astype(np.uint8))
                rle = mask_utils.encode(mask)
                rle["counts"] = rle["counts"].decode("ascii")
                predictions.append({
                    "image_id": rec["image_id"],
                    "category_id": 1,   # competition category — NOT the Detectron2 0-index
                    "segmentation": {"size": rle["size"], "counts": rle["counts"]},
                    "score": float(scores[i]),
                })
    return predictions

test_predictions = run_inference_on_test(test_records, eval_model, SCORE_THRESH, MAX_DETS_PER_IMAGE)
print(f"Generated {len(test_predictions)} predictions across {len(test_records)} test images.")
print(f"(SCORE_THRESH={SCORE_THRESH}, MAX_DETS_PER_IMAGE={MAX_DETS_PER_IMAGE})")

In [ ]:
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission_rgb_baseline.json")
COCO_UTILS_PATH = "tools/coco_utils.py"

if os.path.exists(COCO_UTILS_PATH):
    import sys
    sys.path.insert(0, os.path.dirname(COCO_UTILS_PATH) or ".")
    import coco_utils
    print("tools/coco_utils.py found — check its function names and call the appropriate save/export helper,")
    print("e.g. coco_utils.save_submission(test_predictions, SUBMISSION_PATH), adjusting to match the actual API.")
else:
    print("tools/coco_utils.py not found — writing submission JSON directly.")
    with open(SUBMISSION_PATH, "w") as f:
        json.dump(test_predictions, f)

print("Submission written to:")
print(SUBMISSION_PATH)

In [ ]:
VALIDATOR_PATH = "tools/validate_submission.py"

if os.path.exists(VALIDATOR_PATH):
    !python {VALIDATOR_PATH} --submission {SUBMISSION_PATH}
else:
    print("No tools/validate_submission.py found in this environment — skipping automatic validation.")

print()

In [ ]:
from google.colab import files
import os

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission_rgb_baseline.json")

if os.path.exists(SUBMISSION_PATH):
    print("Downloading your submission file...")
    files.download(SUBMISSION_PATH)
else:
    print(f"File not found at {SUBMISSION_PATH}. Make sure you ran the test inference cell first!")